In [8]:
# ==========================================
# CELL 1: IMPORTS, CONFIGURATION & BACKUP
# ==========================================
import os
import shutil
import datetime
import numpy as np
import joblib

from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA PATHS ───────────────────────────────────────────
# FIRST TRAINING:
# X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
# Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# RETRAINING:
# after running your combiner script, point these to the combined files
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ──────────────────────────────────────────
MODEL_SAVE_PATH = r"C:\Users\kalig\OneDrive\Desktop\tissue_density_model_hybrid.joblib"
MODEL_BACKUP_DIR = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

RANDOM_STATE = 42
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== HYBRID ORDINAL ADABOOST CONFIGURATION ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

if os.path.exists(MODEL_SAVE_PATH):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(
        MODEL_BACKUP_DIR,
        f"tissue_density_model_hybrid_{timestamp}.joblib"
    )
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f" Existing model backed up to: {backup_path}")
else:
    print("No existing model found — fresh training run.")

=== HYBRID ORDINAL ADABOOST CONFIGURATION ===
Feature file : C:\Users\kalig\OneDrive\Desktop\train_X.npy
Label file   : C:\Users\kalig\OneDrive\Desktop\train_y.npy
Model output : C:\Users\kalig\OneDrive\Desktop\tissue_density_model_hybrid.joblib
 Existing model backed up to: C:\Users\kalig\OneDrive\Desktop\model_backups\tissue_density_model_hybrid_20260611_155806.joblib


In [9]:
# ==========================================
# CELL 2: LOAD, SPLIT & BALANCE DATA
# ==========================================
print("\nLoading feature matrix and labels...")
X = np.load(X_PATH)
y = np.load(Y_PATH)

print(f"Data loaded successfully! Shape: X={X.shape}, y={y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# Split first so the test set stays untouched
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nBefore SMOTE — training distribution: {Counter(y_train)}")

smote = SMOTE(
    sampling_strategy='auto',
    random_state=RANDOM_STATE,
    k_neighbors=7
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — training distribution: {Counter(y_train_balanced)}")

real_train_images = X_train.shape[0]
total_train_images = X_train_balanced.shape[0]
synthetic_images = total_train_images - real_train_images
test_images = X_test.shape[0]

print(f"\n{'='*52}")
print("DATA SUMMARY")
print(f"{'='*52}")
print(f"Original training images : {real_train_images}")
print(f"Synthetic images added   : {synthetic_images}")
print(f"Total training images    : {total_train_images}")
print(f"Test images              : {test_images}")
print(f"Features per image       : {X_train_balanced.shape[1]}")
print(f"{'='*52}")


Loading feature matrix and labels...
Data loaded successfully! Shape: X=(9999, 1792), y=(9999,)
Full dataset class distribution: {np.int64(1): np.int64(1172), np.int64(2): np.int64(4282), np.int64(3): np.int64(3991), np.int64(4): np.int64(554)}

Before SMOTE — training distribution: Counter({np.int64(2): 3425, np.int64(3): 3193, np.int64(1): 938, np.int64(4): 443})
After SMOTE — training distribution: Counter({np.int64(1): 3425, np.int64(2): 3425, np.int64(3): 3425, np.int64(4): 3425})

DATA SUMMARY
Original training images : 7999
Synthetic images added   : 5701
Total training images    : 13700
Test images              : 2000
Features per image       : 1792


In [10]:
# ==========================================
# CELL 3: HYBRID ORDINAL ADABOOST
# ==========================================
class HybridOrdinalAdaBoost:
    """
    Hybrid ordinal AdaBoost using K-1 threshold models.

    Decoder A: soft voting
    Decoder B: probability reconstruction from threshold probabilities
    Final: use probability decoder only when stable, else fallback to soft voting
    """
    def __init__(
        self,
        n_estimators=150,
        random_state=None,
        middle_mass_min=0.30,
        winner_margin_min=0.20,
        monotonic_tol=0.01
    ):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.middle_mass_min = middle_mass_min
        self.winner_margin_min = winner_margin_min
        self.monotonic_tol = monotonic_tol
        self.models = []
        self.classes_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        self.classes_ = np.sort(np.unique(y))
        K = len(self.classes_)

        if K < 3:
            raise ValueError(f"HybridOrdinalAdaBoost requires at least 3 ordered classes, got {K}.")

        self.models = []

        for i in range(K - 1):
            threshold = self.classes_[i]
            y_binary = (y > threshold).astype(int)

            model = AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1),
                n_estimators=self.n_estimators,
                random_state=self.random_state
            )
            model.fit(X, y_binary)
            self.models.append(model)

            print(f"✅ Binary classifier {i+1}/{K-1} trained (threshold task: y > {threshold})")

    def predict_votes(self, X: np.ndarray) -> np.ndarray:
        votes = np.column_stack([m.predict(X) for m in self.models])
        vote_sum = votes.sum(axis=1).astype(int)
        vote_sum = np.clip(vote_sum, 0, len(self.classes_) - 1)
        return self.classes_[vote_sum]

    def predict_threshold_proba_raw(self, X: np.ndarray) -> np.ndarray:
        return np.column_stack([m.predict_proba(X)[:, 1] for m in self.models])

    def predict_threshold_proba(self, X: np.ndarray) -> np.ndarray:
        q = self.predict_threshold_proba_raw(X).copy()
        for i in range(1, q.shape[1]):
            q[:, i] = np.minimum(q[:, i - 1], q[:, i])
        return q

    def predict_class_proba(self, X: np.ndarray):
        q_raw = self.predict_threshold_proba_raw(X)
        q_fixed = q_raw.copy()

        for i in range(1, q_fixed.shape[1]):
            q_fixed[:, i] = np.minimum(q_fixed[:, i - 1], q_fixed[:, i])

        n = X.shape[0]
        K = len(self.classes_)
        probs = np.zeros((n, K), dtype=float)

        probs[:, 0] = 1.0 - q_fixed[:, 0]
        for i in range(1, K - 1):
            probs[:, i] = q_fixed[:, i - 1] - q_fixed[:, i]
        probs[:, K - 1] = q_fixed[:, K - 2]

        probs_unclipped = probs.copy()

        probs = np.clip(probs, 0.0, 1.0)
        row_sums = probs.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        probs /= row_sums

        return probs, q_raw, q_fixed, probs_unclipped

    def _prob_decoder_is_reliable(self, probs, q_raw, probs_unclipped):
        negative_mass = (probs_unclipped < -1e-9).any(axis=1)

        monotonic_violation = np.zeros(q_raw.shape[0], dtype=bool)
        for i in range(1, q_raw.shape[1]):
            monotonic_violation |= (q_raw[:, i] - q_raw[:, i - 1]) > self.monotonic_tol

        middle_mass = probs[:, 1] + probs[:, 2]
        low_middle_mass = middle_mass < self.middle_mass_min

        sorted_probs = np.sort(probs, axis=1)
        winner_margin = sorted_probs[:, -1] - sorted_probs[:, -2]
        low_margin = winner_margin < self.winner_margin_min

        reliable = ~(negative_mass | monotonic_violation | low_middle_mass | low_margin)
        return reliable

    def predict_proba_only(self, X: np.ndarray) -> np.ndarray:
        probs, _, _, _ = self.predict_class_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]

    def predict(self, X: np.ndarray) -> np.ndarray:
        vote_preds = self.predict_votes(X)
        probs, q_raw, _, probs_unclipped = self.predict_class_proba(X)
        prob_preds = self.classes_[np.argmax(probs, axis=1)]

        reliable = self._prob_decoder_is_reliable(probs, q_raw, probs_unclipped)
        final_preds = np.where(reliable, prob_preds, vote_preds)
        return final_preds

    def predict_debug(self, X: np.ndarray):
        vote_preds = self.predict_votes(X)
        probs, q_raw, q_fixed, probs_unclipped = self.predict_class_proba(X)
        prob_preds = self.classes_[np.argmax(probs, axis=1)]
        reliable = self._prob_decoder_is_reliable(probs, q_raw, probs_unclipped)
        hybrid_preds = np.where(reliable, prob_preds, vote_preds)

        return {
            "hybrid_preds": hybrid_preds,
            "vote_preds": vote_preds,
            "prob_preds": prob_preds,
            "class_probs": probs,
            "threshold_probs_raw": q_raw,
            "threshold_probs_fixed": q_fixed,
            "prob_decoder_reliable": reliable
        }

In [11]:
# ==========================================
# CELL 4: TRAIN MODEL
# ==========================================
print("\nTraining Hybrid Ordinal AdaBoost...")
model = HybridOrdinalAdaBoost(
    n_estimators=150,
    random_state=RANDOM_STATE,
    middle_mass_min=0.30,
    winner_margin_min=0.20,
    monotonic_tol=0.01
)
model.fit(X_train_balanced, y_train_balanced)
print("✅ Training complete!")


Training Hybrid Ordinal AdaBoost...
✅ Binary classifier 1/3 trained (threshold task: y > 1)
✅ Binary classifier 2/3 trained (threshold task: y > 2)
✅ Binary classifier 3/3 trained (threshold task: y > 3)
✅ Training complete!


In [12]:
# ==========================================
# CELL 5: EVALUATE ALL 3 DECODERS
# ==========================================
print("\nEvaluating on test set...")

debug = model.predict_debug(X_test)

y_pred_hybrid = debug["hybrid_preds"]
y_pred_vote   = debug["vote_preds"]
y_pred_prob   = debug["prob_preds"]

acc_hybrid = accuracy_score(y_test, y_pred_hybrid)
acc_vote   = accuracy_score(y_test, y_pred_vote)
acc_prob   = accuracy_score(y_test, y_pred_prob)

cm_hybrid = confusion_matrix(y_test, y_pred_hybrid)

print(f"\n{'='*56}")
print("PERFORMANCE REPORT")
print(f"{'='*56}")
print(f"Images model trained on : {total_train_images}")
print(f"  ↳ Real images         : {real_train_images}")
print(f"  ↳ SMOTE synthetic     : {synthetic_images}")
print(f"Images tested on        : {test_images}")
print(f"Soft-vote accuracy      : {acc_vote:.4f}")
print(f"Probability accuracy    : {acc_prob:.4f}")
print(f"Hybrid accuracy         : {acc_hybrid:.4f}")
print(f"{'='*56}")

reliable_mask = debug["prob_decoder_reliable"]
num_reliable = int(reliable_mask.sum())
num_fallback = int((~reliable_mask).sum())

print(f"\nProbability decoder trusted on : {num_reliable} samples")
print(f"Fallback to soft voting on     : {num_fallback} samples")

print("\nPer-class breakdown (HYBRID):")
print(classification_report(
    y_test,
    y_pred_hybrid,
    target_names=[
        'Tissue Density Category 1',
        'Tissue Density Category 2',
        'Tissue Density Category 3',
        'Tissue Density Category 4'
    ]
))

print("Confusion Matrix (HYBRID):")
header = "              " + "  ".join(f"Pred {i+1}" for i in range(4))
print(header)
for i, row in enumerate(cm_hybrid):
    print(f"Actual {i+1}     " + "  ".join(f"{v:6d}" for v in row))

print("\nFirst 5 raw threshold probabilities:")
print(debug["threshold_probs_raw"][:5])

print("\nFirst 5 fixed threshold probabilities:")
print(debug["threshold_probs_fixed"][:5])

print("\nFirst 5 reconstructed class probabilities:")
print(debug["class_probs"][:5])

print("\nFirst 20 gate decisions (True = prob decoder trusted):")
print(debug["prob_decoder_reliable"][:20])


Evaluating on test set...

PERFORMANCE REPORT
Images model trained on : 13700
  ↳ Real images         : 7999
  ↳ SMOTE synthetic     : 5701
Images tested on        : 2000
Soft-vote accuracy      : 0.6685
Probability accuracy    : 0.1720
Hybrid accuracy         : 0.6685

Probability decoder trusted on : 0 samples
Fallback to soft voting on     : 2000 samples

Per-class breakdown (HYBRID):
                           precision    recall  f1-score   support

Tissue Density Category 1       0.54      0.71      0.61       234
Tissue Density Category 2       0.73      0.66      0.69       857
Tissue Density Category 3       0.74      0.66      0.70       798
Tissue Density Category 4       0.37      0.66      0.47       111

                 accuracy                           0.67      2000
                macro avg       0.59      0.67      0.62      2000
             weighted avg       0.69      0.67      0.67      2000

Confusion Matrix (HYBRID):
              Pred 1  Pred 2  Pred 3  Pred

In [13]:
# ==========================================
# CELL 6: SAVE MODEL
# ==========================================
print(f"\nSaving model to {MODEL_SAVE_PATH}...")
joblib.dump(model, MODEL_SAVE_PATH)
print(" Done! Hybrid ordinal model saved.")


Saving model to C:\Users\kalig\OneDrive\Desktop\tissue_density_model_hybrid.joblib...
 Done! Hybrid ordinal model saved.


In [14]:
# ==========================================
# CELL 7: PREDICTION UTILITY
# ==========================================
def predict_new(X_new: np.ndarray):
    """
    Returns:
      hybrid_preds       -> final hybrid predictions
      vote_preds         -> soft-vote predictions
      prob_preds         -> probability-decoder predictions
      class_probs        -> reconstructed class probabilities
      threshold_probs    -> monotonic threshold probabilities
      trusted_prob_mask  -> whether probability decoder was trusted
    """
    debug = model.predict_debug(X_new)
    return (
        debug["hybrid_preds"],
        debug["vote_preds"],
        debug["prob_preds"],
        debug["class_probs"],
        debug["threshold_probs_fixed"],
        debug["prob_decoder_reliable"]
    )

print(" Prediction utility ready.")
print("Use:")
print("hybrid_preds, vote_preds, prob_preds, class_probs, threshold_probs, trusted_mask = predict_new(X_new)")

 Prediction utility ready.
Use:
hybrid_preds, vote_preds, prob_preds, class_probs, threshold_probs, trusted_mask = predict_new(X_new)
